In [3]:
"""Generate an offline interactive 3D visual. Run from any directory with Python 3."""

# lets create a visuals for gradient descent for a loss function 

#lets first define the dataset X,Y from sklearn.datasets import make_moons
from sklearn.datasets import make_moons
X, Y = make_moons(n_samples=200, noise=0.3, random_state=42)
#lets create a neural network with 2 hidden layers and 10 neurons in each layer
import numpy as np
#lets save the gradients in a list for each epoch

class NeuralNetwork:
    def __init__(self, input_size, hidden_size, output_size):
        self.W1 = np.random.randn(input_size, hidden_size)
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, output_size)
        self.b2 = np.zeros((1, output_size))
    def forward(self, X):
        self.Z1 = np.dot(X, self.W1) + self.b1
        self.A1 = self.sigmoid(self.Z1)
        self.Z2 = np.dot(self.A1, self.W2) + self.b2
        self.A2 = self.sigmoid(self.Z2)
        return self.A2
    def backward(self, X, Y, output):
        m = Y.shape[0]
        dZ2 = output - Y
        dW2 = np.dot(self.A1.T, dZ2) / m
        db2 = np.sum(dZ2, axis=0, keepdims=True) / m
        dZ1 = np.dot(dZ2, self.W2.T) * self.sigmoid_derivative(self.A1)
        dW1 = np.dot(X.T, dZ1) / m
        db1 = np.sum(dZ1, axis=0, keepdims=True) / m
        return dW1, db1, dW2, db2
    def update_parameters(self, dW1, db1, dW2, db2, learning_rate):
        self.W1 -= learning_rate * dW1
        self.b1 -= learning_rate * db1
        self.W2 -= learning_rate * dW2
        self.b2 -= learning_rate * db2
    def sigmoid(self, Z):
        return 1 / (1 + np.exp(-Z))
    def sigmoid_derivative(self, Z):
        return Z * (1 - Z)


In [4]:


#lets fit the neural network to the dataset and save the gradients for each epoch keep the network small so that we can visualize the gradients in 3D
nn = NeuralNetwork(input_size=2, hidden_size=10, output_size=1)
gradients = []
for epoch in range(1000):
    output = nn.forward(X)
    print(f"Epoch {epoch}, Output: {output.flatten()}")
    #save the gradients for each epoch lets save the Weights and biases gradients as well as the epoch number in a list of lists

    dW1, db1, dW2, db2 = nn.backward(X, Y.reshape(-1, 1), output)
    W1_norm = np.linalg.norm(nn.W1)
    W2_norm = np.linalg.norm(nn.W2)
    loss = np.mean((output - Y.reshape(-1, 1)) ** 2)
    accuracy = np.mean((output > 0.5) == Y.reshape(-1, 1))
    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss}, Accuracy: {accuracy}")
    gradients.append([epoch, dW1, db1, dW2, db2, W1_norm, W2_norm, loss])
    nn.update_parameters(dW1, db1, dW2, db2, learning_rate=0.01)

#lets plot the gradients in 3D using plotly
#import plotly and use plotly to visualize the gradients in 3D



Epoch 0, Output: [0.59980303 0.09569366 0.11840391 0.09136207 0.23510073 0.16340877
 0.43791151 0.06231036 0.04005069 0.24815494 0.358767   0.04898554
 0.06419222 0.51426287 0.04720218 0.04300091 0.20441769 0.1091143
 0.51924481 0.32166109 0.16059967 0.11695299 0.12597709 0.43486993
 0.34063231 0.40732995 0.40087444 0.31854972 0.24036355 0.22836863
 0.07211778 0.08558064 0.17199603 0.52165682 0.11800121 0.27539573
 0.1436086  0.14977402 0.06244031 0.520862   0.41447713 0.0419097
 0.19657011 0.10265349 0.15428968 0.05516334 0.11757589 0.11436459
 0.11051765 0.40931151 0.1646115  0.05468001 0.12989973 0.17288011
 0.21711729 0.50974298 0.14243181 0.62406312 0.04522815 0.50931357
 0.0372504  0.4418822  0.18303885 0.07094174 0.20592236 0.29765317
 0.24990488 0.04845176 0.42764466 0.11993655 0.28183908 0.21830227
 0.06708799 0.22832823 0.32713081 0.40333211 0.11857816 0.14373009
 0.25126137 0.23822474 0.09374052 0.40318278 0.30089359 0.05275683
 0.03910636 0.47951717 0.08443775 0.15651195 0.

In [ ]:

import plotly.graph_objects as go
#lets take two gradients dW1 and dW2 and plot them in 3D with epoch on the x-axis, dW1 on the y-axis and dW2 on the z-axis
epochs = [g[0] for g in gradients]
dW1 = [np.linalg.norm(g[1]) for g in gradients]
dW2 = [np.linalg.norm(g[3]) for g in gradients]
W1_norm = [g[5] for g in gradients]
W2_norm = [g[6] for g in gradients]
loss = [g[7] for g in gradients]
#lets create a 3D plot with onw axis as loss function one axis W1 and one axis W2 and color the points by epoch
print("Creating 3D plot...")
#also show gradients as arrows in the 3D plot
fig = go.Figure(data=[go.Scatter3d(
    x=loss,
    y=W1_norm,
    z=W2_norm,
    mode='markers',
    marker=dict(
        size=5,
        color=epochs,                # set color to an array/list of desired values
        colorscale='Viridis',   # choose a colorscale
        opacity=0.8
    )
)])
#add arrows to the plot to show the gradients
for i in range(len(epochs)):
    fig.add_trace(go.Cone(
        x=[loss[i]],
        y=[W1_norm[i]],
        z=[W2_norm[i]],
        u=[dW1[i]],
        v=[dW2[i]],
        w=[0],
        sizemode="absolute",
        sizeref=0.1,
        anchor="tail",
        showscale=False
    ))
fig.update_layout(
    scene=dict(
        xaxis_title='Loss',
        yaxis_title='||W1||',
        zaxis_title='||W2||'
    ),
    title='3D Visualization of Gradients during Training'
)
#save the plot as an html file
fig.write_html("gradients_3d_plot.html")    

#plot the loss function over epochs
import matplotlib.pyplot as plt
plt.plot(epochs, loss)
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Loss over Epochs')
plt.show()
# plt.savefig("loss_over_epochs.png")